Run the following cell only once: install nirfasterff and download the necessary sample data.
You may use the same code if you are running local. The correct version of nirfasterff should be automatically determined.

In [ ]:
import platform
import subprocess

version_string = platform.python_version()
major_ver = version_string.split('.')[1]
cpulib_name = 'cpu-linux-python3'+major_ver+'.zip'
gpulib_name = 'gpu-linux-python3'+major_ver+'.zip'
!git clone https://github.com/milabuob/nirfaster-FF.git
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/Conte69_on_TT_32k.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/IM_Gordon_2014_333_Parcels.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol_optodes_96x92.mat

current_os = platform.system()
if current_os == "Darwin":
    cpulib_name = 'cpu-mac-python3'+major_ver+'.zip'
else:
    if current_os == "Windows":
        cpulib_name = 'cpu-win-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-win-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
    elif current_os == "Linux":
        cpulib_name = 'cpu-linux-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-linux-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
        
subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+cpulib_name])
subprocess.run(['unzip', cpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
subprocess.run(['rm', cpulib_name])

!pip install ipympl
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import sys
import os
sys.path.insert(1, 'nirfaster-FF/')
import nirfasterff as ff # ff is short for fast and furious
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import copy
import math
# enable interactive matplotlib
# comment out the following lines if you run local
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib ipympl

In [ ]:
# load the 3D volume
# Note that the data type must be uint8
vol = sio.loadmat('headvol.mat')['mask']

# Load the optode information. This is from the WashU HD-DOT 96 sources and 92 detectors, as from NeuroDOT
tmp = sio.loadmat('headvol_optodes_96x92.mat')
link = tmp['link']
srcpos = tmp['srcpos']
detpos = tmp['detpos']

In [ ]:
# defining optical properties
# mask above has 5 regions, labelled as
# 1: csf; 2: white; 3: gray; 4: bone; 5: skin

# from this paper, we have tissue specific HbO, Hb and scattering properties: https://doi.org/10.1117/1.NPh.8.1.015013
# Region    	HbO (mM)	Hb (mM)	Sa (mm−1)	Sp
# Skin	        0.0575	    0.0313	0.53	    1.15
# Skull	        0.0443	    0.0195	0.72	    0.89
# CSF	        0.0110	    0.0083	0.30	    0
# Gray matter	0.0559	    0.0350	0.50	    1.73
# White matter	0.0680	    0.0265	0.81	    1.31
# let's define a chromophore matrix
C = np.array([[0.0575,  0.0313],
            [0.0443,    0.0195],
            [0.0110,    0.0083],
            [0.0559,    0.0350],
            [0.0680,    0.0265]])

# assuming two wavelength of 760 and 850 nm, from NIRFAST we gave these extinction coeff
# wavelenth(nm) HbO(mm^-1/mM)   HbR(mm^-1/mM)
# 760           1.3495580e-01   3.5662416e-01
# 850           2.4365740e-01   1.5921100e-01
# let's define an extinction coefficient matrix
E = np.array([[1.3495580e-01,   3.5662416e-01],
              [2.4365740e-01,   1.5921100e-01]])

# we can now calculate mua at each wavelength using Beer's law
mua = np.dot(C,E)

# we can calulate musp using Mie's theory
Sa = np.array([0.53, 0.72, 0.3, 0.5, 0.81])
Sp = np.array([1.15, 0.89, 0, 1.73, 1.31])
musp = np.zeros((5,2))
wv = np.array([760, 850])
musp[:,0] = Sa * (wv[0] / 1000) ** (-Sp)
musp[:,1] = Sa * (wv[1] / 1000) ** (-Sp)

print(f"mua(mm-1) = 760 nm; 850 nm\n{mua}")
print(f"musp(mm-1) = 760 nm; 850 nm\n{musp}")

In [ ]:
# define the parameters for the mesh and the optodes
params = ff.utils.MeshingParams()
# add an offset so that the final mesh is in MNI space/coordinates
params.offset = np.array([-87., -124.,  -76.5])
# this defines mesh resolution: low values (0.5-1) are high mesh density, high values (5-10) are low mesh density. 
# for an adult head, to ensure numerical stability, we often have an average mesh element volume of ~1 mm^2
# for purpose of this excercise, volumes of 2-3 mm^3 are OK
params.facet_distance = 2
params.facet_size = 2
params.general_cell_size = 2

# turn off default smoothing as for high resolution meshes, it often does not converge
params.smooth = False

# from calculated mua and musp above, we will set optical properties for 760 nm, with refractive index of 1.4
# 1: csf; 2: white; 3: gray; 4: bone; 5: skin
optical_props = np.array([[1, 0.0035, 0.3, 1.4],
                        [2, 0.0156, 1.16, 1.4],
                        [3, 0.0161, 0.80, 1.4],
                        [4, 0.0107, 0.92, 1.4],
                        [5, 0.0154, 0.73, 1.4]])

# optodes
sources = ff.base.optode(srcpos)
detectors = ff.base.optode(detpos)

# create the mesh
mesh = ff.base.stndmesh()
mesh.from_volume(vol, param=params, prop=optical_props, src=sources, det=detectors, link=link)

In [ ]:
# Display basic information about the mesh
print(f"Number of nodes = {mesh.nodes.shape[0]}")
print(f"Number of elements = {mesh.elements.shape[0]}")
print(f"Average element volume = {np.mean(mesh.element_area)} +/- {np.std(mesh.element_area)}")

These numbers might be slightly different depending on the machine you run the code on. ~350k nodes is to be considered intermediately dense for a head mesh. 
Let's have a quick look at the quality of the mesh: the mesh element volumes should follow a Gaussian-ish distribution

In [ ]:
# plot histogran of mesh element volume
plt.close('all')
plt.hist(mesh.element_area,100)
plt.xlabel('Element volume (mm$^3$)')
plt.ylabel('Number of elements')
plt.title('Element Volume Histogram')
plt.show()

In [ ]:
# save the mesh as to not having to create it each time.
os.makedirs('meshes', exist_ok=True)
mesh.save_nirfast('meshes/headmesh_760nm')

# remember from above, we also calculated mua and musp for 850 nm, so now change the values and save a new mesh!
mesh.set_prop(np.array([[1, 0.005, 0.3, 1.4],
                       [2, 0.028, 1.00, 1.4],
                       [3, 0.026, 0.66, 1.4],
                       [4, 0.019, 0.83, 1.4],
                       [5, 0.025, 0.64, 1.4]]))
mesh.save_nirfast('meshes/headmesh_850nm')